In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType

#Reading From Silver Table

In [0]:
query = """
SELECT 
    T.transaction_id,
    D.date_key,
    PC.promo_key,
    T.order_date,
    S.store_key,
    T.store_id,
    T.store_name,
    T.region,
    P.product_key,
    T.product_id,
    T.product_name,
    T.category,
    T.quantity,
    T.applied_unit_price,
    T.applied_discount,
    T.customer_id,
    T.loyalty_member,
    ROUND(
        CASE 
            WHEN T.applied_discount IS NULL THEN T.applied_unit_price*T.quantity
            ELSE (1-T.applied_discount)*T.applied_unit_price*T.quantity
        END, 2) AS revenue
FROM silver.transtractions T
LEFT JOIN gold.dim_store S
    ON T.store_id = S.store_id
LEFT JOIN gold.dim_product P
    ON T.product_id = P.product_id
LEFT JOIN gold.dim_promo_calander PC
    ON T.order_date BETWEEN PC.start_date AND PC.end_date
    AND T.store_id = PC.store_id
    AND T.category = PC.category
LEFT JOIN gold.dim_date D
    ON T.order_date = D.full_day
"""

df = spark.sql(query)

In [0]:
df.display()

#Write into gold layer

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_tanstractions")
)

In [0]:
%sql
SELECT * FROM gold.fact_tanstractions